In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("login", "")
dbutils.widgets.text("target_catalog", "")
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("container", "")

login           = dbutils.widgets.get("login")
catalog         = dbutils.widgets.get("target_catalog")
storage_account = dbutils.widgets.get("storage_account")
container       = dbutils.widgets.get("container")

assert all([login, catalog, storage_account, container])

bronze = f"{login}_bronze"
abfss  = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

In [0]:
spark.sql(f"""CREATE EXTERNAL VOLUME IF NOT EXISTS {catalog}.{bronze}.landing
              LOCATION '{abfss}/landing'""")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{bronze}.checkpoints")
landing = f"/Volumes/{catalog}/{bronze}/landing"
chk     = f"/Volumes/{catalog}/{bronze}/checkpoints"
print("landing:", landing)

In [0]:
def ingest(dataset, fmt, filename, opts=None):
    target = f"{catalog}.{bronze}.{dataset}_bronze"
    r = (spark.readStream.format("cloudFiles")
         .option("cloudFiles.format", fmt)
         .option("cloudFiles.schemaLocation", f"{chk}/{dataset}/schema")
         .option("pathGlobFilter", filename))
    
    for k, v in (opts or {}).items():
        r = r.option(k, v)

    (r.load(landing)
       .withColumn("source_file",  F.col("_metadata.file_name"))
       .withColumn("ingestion_ts", F.current_timestamp())
       .withColumn("load_date",    F.current_date())
       .writeStream
       .option("checkpointLocation", f"{chk}/{dataset}/checkpoint")
       .trigger(availableNow=True)
       .toTable(target)
    ).awaitTermination()

    print(target, "->", spark.table(target).count(), "rows")

ingest("raw_events", "json", "raw_events.jsonl")
ingest("qnap_stats", "json", "qnap_stats.jsonl")
ingest("camera_dim", "csv",  "camera_dim.csv", {"header": "true"})

In [0]:
display(spark.table(f"{catalog}.{bronze}.raw_events_bronze")
        .select("source_file","ingestion_ts","load_date","*").limit(5))

###Test assuring indepotency

In [0]:
def assert_idempotent(dataset, fmt, filename, opts=None):
    table = f"{catalog}.{bronze}.{dataset}_bronze"
    before = spark.table(table).count()
    ingest(dataset, fmt, filename, opts)
    after = spark.table(table).count()
    assert before == after, f"{table} not idempotent: {before} -> {after}"

assert_idempotent("raw_events", "json", "raw_events.jsonl")